In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer # BertTokenizer is imported but not used in this file, kept for reference if needed elsewhere.


class BERTEncoder(nn.Module):
    """
    Encodes input text using a pre-trained BERT model.
    It specifically extracts the [CLS] token representation for sentence-level features.
    """
    def __init__(self, pretrained_model='bert-base-uncased'):
        super(BERTEncoder, self).__init__()
        # Load the pre-trained BERT model.
        # This will download the weights if not already cached.
        self.bert = BertModel.from_pretrained(pretrained_model)
        # Ensure that the output dimension of BERT matches the expected hidden_size
        # For 'bert-base-uncased', config.hidden_size is 768.
        self.hidden_size = self.bert.config.hidden_size


    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Processes input_ids and attention_mask through BERT to get the [CLS] token embedding.

        Args:
            input_ids (torch.Tensor): Tensor of input token IDs for a sentence.
                                      Shape: (batch_size, sequence_length)
            attention_mask (torch.Tensor): Tensor indicating which tokens are not padding.
                                           Shape: (batch_size, sequence_length)

        Returns:
            torch.Tensor: The [CLS] token representation for each sentence in the batch.
                          Shape: (batch_size, hidden_size)
        """
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # The [CLS] token is typically the first token (index 0) in the sequence.
        # Its hidden state from the last layer is used as a summary of the sequence.
        cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output


class A2CExtractorAgent(nn.Module):
    """
    An A2C (Advantage Actor-Critic) agent designed for sentence extraction.
    It uses a BERT encoder to get sentence representations, then predicts
    whether to select or skip a sentence (actor) and estimates its value (critic).
    """
    def __init__(self, hidden_size: int = 768, num_actions: int = 2):
        """
        Initializes the A2C Extractor Agent.

        Args:
            hidden_size (int): The dimension of the BERT encoder's output.
                               (e.g., 768 for bert-base-uncased).
            num_actions (int): Number of possible actions (e.g., 2 for select/skip).
        """
        super(A2CExtractorAgent, self).__init__()
        self.encoder = BERTEncoder(pretrained_model='bert-base-uncased')

        # Ensure hidden_size matches the actual output dimension of the BERT encoder
        # This makes the model more robust if the BERTEncoder's default changes or is overridden
        if self.encoder.hidden_size != hidden_size:
            print(f"Warning: Provided hidden_size ({hidden_size}) does not match BERTEncoder's output hidden_size ({self.encoder.hidden_size}). Using encoder's hidden_size.")
            hidden_size = self.encoder.hidden_size

        # Actor network: predicts the probability of each action (select/skip)
        self.actor = nn.Linear(hidden_size, num_actions)

        # Critic network: estimates the value of the current state (sentence)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        """
        Performs a forward pass through the encoder, actor, and critic networks.

        Args:
            input_ids (torch.Tensor): Token IDs for a batch of sentences.
                                      Shape: (batch_size, sequence_length)
            attention_mask (torch.Tensor): Attention masks for the batch of sentences.
                                           Shape: (batch_size, sequence_length)

        Returns:
            Tuple[torch.Tensor, torch.Tensor]:
                - action_logits (torch.Tensor): Logits for each action (e.g., [log_prob_skip, log_prob_select]).
                                                Shape: (batch_size, num_actions)
                - state_value (torch.Tensor): Estimated value of the current state.
                                              Shape: (batch_size, 1)
        """
        # Get sentence representation from BERT encoder
        x = self.encoder(input_ids, attention_mask)

        # Calculate action logits and state value
        action_logits = self.actor(x)
        state_value = self.critic(x)
        return action_logits, state_value

    def act(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        """
        Selects an action based on the current state (sentence) and provides
        the necessary values for A2C training.

        Args:
            input_ids (torch.Tensor): Token IDs for a batch of sentences (or a single sentence).
                                      Shape: (batch_size, sequence_length)
            attention_mask (torch.Tensor): Attention masks for the batch of sentences.
                                           Shape: (batch_size, sequence_length)

        Returns:
            Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
                - action (torch.Tensor): The sampled action (0 for skip, 1 for select).
                                         Shape: (batch_size,)
                - log_prob (torch.Tensor): Log probability of the sampled action.
                                           Shape: (batch_size,)
                - entropy (torch.Tensor): Entropy of the action distribution.
                                          Shape: (batch_size,)
                - value (torch.Tensor): Estimated value of the state.
                                        Shape: (batch_size, 1)
        """
        logits, value = self.forward(input_ids, attention_mask)

        # Convert logits to probabilities using softmax
        probs = F.softmax(logits, dim=-1)

        # Create a categorical distribution from probabilities
        dist = torch.distributions.Categorical(probs)

        # Sample an action from the distribution
        action = dist.sample()

        # Calculate log probability of the sampled action
        log_prob = dist.log_prob(action)

        # Calculate the entropy of the distribution (for exploration)
        entropy = dist.entropy()

        return action, log_prob, entropy, value

# Example Usage (for testing the module)
if __name__ == '__main__':
    # Initialize tokenizer for creating dummy inputs
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    # Example sentence
    sentence1 = "This is a test sentence for the extractor agent."
    sentence2 = "The quick brown fox jumps over the lazy dog."

    # Tokenize the sentences
    inputs = tokenizer([sentence1, sentence2], return_tensors='pt', padding='max_length', truncation=True, max_length=512)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    print(f"Input IDs shape: {input_ids.shape}")
    print(f"Attention Mask shape: {attention_mask.shape}")

    # Initialize the A2C Extractor Agent
    agent = A2CExtractorAgent()
    print(f"Agent initialized with BERT encoder output hidden_size: {agent.encoder.hidden_size}")

    # Perform a forward pass
    logits, value = agent.forward(input_ids, attention_mask)
    print(f"\nForward Pass Output:")
    print(f"Action Logits shape: {logits.shape}") # Should be (batch_size, num_actions=2)
    print(f"State Value shape: {value.shape}")   # Should be (batch_size, 1)

    # Use the act method to sample an action
    action, log_prob, entropy, sampled_value = agent.act(input_ids, attention_mask)
    print(f"\nAct Method Output (Sampled Action):")
    print(f"Action: {action}")             # e.g., tensor([0, 1]) where 0=skip, 1=select
    print(f"Log Prob: {log_prob}")         # Log probability of the taken action
    print(f"Entropy: {entropy}")           # Entropy of the policy distribution
    print(f"Sampled Value: {sampled_value}") # Value estimate for the state

    # You can move the model to GPU if available
    if torch.cuda.is_available():
        print("\nMoving model to GPU...")
        agent.to('cuda')
        input_ids = input_ids.to('cuda')
        attention_mask = attention_mask.to('cuda')
        action_cuda, log_prob_cuda, entropy_cuda, value_cuda = agent.act(input_ids, attention_mask)
        print(f"Action on GPU: {action_cuda}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Input IDs shape: torch.Size([2, 512])
Attention Mask shape: torch.Size([2, 512])


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Agent initialized with BERT encoder output hidden_size: 768

Forward Pass Output:
Action Logits shape: torch.Size([2, 2])
State Value shape: torch.Size([2, 1])

Act Method Output (Sampled Action):
Action: tensor([0, 0])
Log Prob: tensor([-0.6334, -0.6793], grad_fn=<SqueezeBackward1>)
Entropy: tensor([0.6912, 0.6930], grad_fn=<NegBackward0>)
Sampled Value: tensor([[-0.4439],
        [-0.1268]], grad_fn=<AddmmBackward0>)
